# NanoJev (zero-shot) vs Random Forest: классификация паттернов поведения

**Задача:** по окну потребления мощности (P_RMS, мВт) умной розетки определить **паттерн поведения** (класс бытового устройства), закодированный в имени файла.

**Что в этом ноутбуке:**
1. Загрузка данных из **Google Drive** (как в исходном `TimesFM3_DeepSeek.ipynb`) — папка `SmartPlug`, окна по 90 отсчётов, метка из имени `plug_dump_..._<LABEL>_<DURATION>.csv`.
2. Классификация через **NanoJev** ([`TianyuCodings/NanoJev`](https://github.com/TianyuCodings/NanoJev)) — открытая «нано-реплика» Jev: Qwen3-0.6B + decision-головы, **zero output-token decoding**: state + вопрос + кандидаты → полные вероятности одним forward.
3. Сравнение с бейзлайном: **Random Forest** на тех же статистических признаках.
4. Итоговая честная таблица на одном и том же тесте (30% стратифицированная выборка).

> Работает и в **Google Colab** (GPU T4+), и в **VS Code**. В Colab данные монтируются из Google Drive, локально — из папки `SmartPlug` (переменная `SMARTPLUG_DIR` или `~/SmartPlug`).

⚠️ **Важные оговорки про zero-shot NanoJev:**
- Модель обучалась на **игровых** задачах (Maze, Snake, ViZDoom Basic / Predict Position), а не на временны́х рядах. Её «state» — это **текст**: строки/словари сериализуются в шаблон `State:\n...\nQuestion...\nCandidate...\nDecision:`. Числовые ряды внутри модели «нет» — мы подаём статистики и прорежённую кривую как текст.
- Это **не** обещание высокого качества: цель раздела — понять, насколько обобщается чекпойнт на полностью новую предметную область (электрическая нагрузка) в сравнении с простым RF на тех же признаках.
- Инференс NanoJev требует **CUDA** (`DecisionPredictor` жёстко проверяет `cuda:0`) и ~2.5 ГБ весов в fp32 + bf16-автокаст.

## 0. Окружение и данные

In [ ]:
# 0.1 Установка зависимостей (NanoJev не ставится как pip-пакет: ниже клонируем репо).
#     transformers >= 4.46 обязателен: чекпойнт NanoJev использует Qwen2-токенизатор с
#     list-стилем extra_special_tokens.
%pip install -q "transformers>=4.46" torch safetensors huggingface_hub numpy pandas scikit-learn matplotlib tqdm
print("зависимости установлены")

In [ ]:
# 0.2 Импорты и общие настройки
import os, sys, re, json, warnings, subprocess
from collections import Counter

os.environ.setdefault("USE_TF", "0")  # чтобы transformers не схватывал TF-рантайм при импорте

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
np.random.seed(42)

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print("Среда:", "Google Colab" if IN_COLAB else "Локальная (VS Code)")
print("Python:", sys.version.split()[0])

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Устройство:", DEVICE, "| CUDA:", torch.cuda.is_available())
except ImportError:
    DEVICE = "cpu"
    print("Устройство: cpu (torch не найден)")

In [ ]:
# 0.3 Папка с данными
#   Colab  -> монтируем Google Drive и берём /content/drive/MyDrive/SmartPlug
#   VS Code-> переменная окружения SMARTPLUG_DIR либо ~/SmartPlug
DATA_DIR = os.environ.get("SMARTPLUG_DIR", "").strip()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DATA_DIR or "/content/drive/MyDrive/SmartPlug"
else:
    DATA_DIR = DATA_DIR or os.path.expanduser("~/SmartPlug")

print("Папка данных:", DATA_DIR)
print("Существует:", os.path.isdir(DATA_DIR))
if not os.path.isdir(DATA_DIR):
    raise SystemExit(
        "Папка с данными не найдена. "
        "В Colab положите датасет в MyDrive/SmartPlug; "
        "локально — укажите SMARTPLUG_DIR или создайте ~/SmartPlug."
    )

## 1. Загрузка данных (как в исходном ноутбуке)

In [ ]:
# 1.1 Параметры разбиения
CHUNK_LENGTH = 90                  # длина одного окна (в отсчётах)
FILTER_OUT = ("IdleCharge", "MixedBrowsing", "Browsing", "VKAudio")  # классы, которые пропускаем
COLUMN = " P_RMS, mW"              # колонка мощности (мощность потребления)

filelist = sorted(f for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv"))
print("Найдено файлов:", len(filelist))
if not filelist:
    raise SystemExit("В папке данных нет .csv-файлов.")

In [ ]:
# 1.2 Парсер имени файла: plug_dump_YYYY_MM_DD_hh_mm_ss_<LABEL>_<DURATION>.csv
def get_label_duration(filename: str):
    duration = re.findall(r"plug_dump_\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2}_\D+?_(\d+)\.csv", filename)
    label = re.findall(r"plug_dump_\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2}_(\D+?)_\d+\.csv", filename)
    if not duration or not label:
        return None, None
    return label[0], duration[0]

# Проверка на примере
sample = filelist[0]
print("Пример имени:", sample)
print("Метка:", get_label_duration(sample))

In [ ]:
# 1.3 Чтение файлов и сбор окон длиной CHUNK_LENGTH
#     (копия логики исходного ноутбука; ищем колонку мощности по strip-имени, чтобы
#      не зависеть от лидирующего пробела в заголовке)
Labels, ndata, skipped = [], None, 0

for filename in tqdm(filelist, desc="Обработка файлов"):
    label, _ = get_label_duration(filename)
    if label is None:
        skipped += 1
        continue
    if any(tag in label for tag in FILTER_OUT):
        continue
    try:
        df = pd.read_csv(os.path.join(DATA_DIR, filename), delimiter=";")
        col = next((c for c in df.columns if c.strip() == COLUMN.strip()), None)
        if col is None:
            skipped += 1
            continue
        series = df[col].to_numpy(dtype=np.float64)
    except Exception:
        skipped += 1
        continue

    num_chunks = len(series) // CHUNK_LENGTH
    for i in range(num_chunks):
        chunk = series[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH]
        ndata = chunk[None, :] if ndata is None else np.vstack([ndata, chunk])
        Labels.append(label)

ndata = np.asarray(ndata, dtype=np.float32) if ndata is not None else np.empty((0, CHUNK_LENGTH))
print("Пропущено файлов:", skipped)
print("Всего фрагментов:", len(Labels), "| Форма массива:", ndata.shape)

print("Метки и их частоты:")
for u, c in Counter(Labels).items():
    print(f"  {u}: {c}")
if len(Labels) < 20:
    raise SystemExit("Слишком мало фрагментов — проверьте папку/фильтры.")

In [ ]:
# 1.4 Кодирование меток и сплит train/test (стратификация), как в исходнике
le = LabelEncoder()
y_encoded = le.fit_transform(Labels)
classes = list(le.classes_)
print("Классов:", len(classes), "| Классы:", ", ".join(classes))

X_train, X_test, y_train, y_test = train_test_split(
    ndata, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
test_labels = le.inverse_transform(y_test)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 2. NanoJev (клон Jev) — классификация паттернов поведения, zero-shot

**Что это такое.** [NanoJev](https://github.com/TianyuCodings/NanoJev) — «нано-реплика» Jev от TypeSafe: Qwen3-0.6B + decision-головы, обученная **полной кросс-энтропией на полные вопросы**. Никакой генерации токенов ответа — каждый request это:

```
State: <номер окна + статистики + прорежённая кривая>
Question type: choice
Question: <инструкция>
Candidate: <ключ>: <описание паттерна>
Decision:  <EOS>
```

Все candidate-пути кодируются и прогоняются **одним forward**; у каждого вектора с последней позиции: LayerNorm → `Linear(hidden, 1)` → скаляр-логит, а для `choice` поверх добавляется **set-attention** (MultiheadAttention по кандидатам одного вопроса + признак `log #candidates`, tanh-остаток). Ответ — `softmax` по кандидатам: полные вероятности без генерации.

**Что подаём.** Числовой чанк в модель «передать нельзя» — state это текст. Строим компактное текстовое описание: те же 11 статистик, что пойдут в Random Forest (честное сравнение на одном входе), плюс прорежённая кривая. Вопрос — `choice` с фактическими классами из данных (как в предыдущем ноутбуке с Laya).

In [ ]:
# 2.1 Клонируем репозиторий NanoJev (только код + данные; веса ниже с HF).
REPO_DIR = "/content/NanoJev" if IN_COLAB else os.path.expanduser("~/NanoJev")
if not os.path.isdir(os.path.join(REPO_DIR, "scripts")):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/TianyuCodings/NanoJev.git", REPO_DIR], check=True)
SCRIPTS_DIR = os.path.join(REPO_DIR, "scripts")
sys.path.insert(0, SCRIPTS_DIR)
print("scripts:", SCRIPTS_DIR)
print("predict_toy_decisions.py найден:", os.path.isfile(os.path.join(SCRIPTS_DIR, "predict_toy_decisions.py")))

In [ ]:
# 2.2 Скачиваем выпущенный чекпойнт unified-games-v1 (best.safetensors + config + tokenizer + backbone_config).
from huggingface_hub import snapshot_download

CKPT_DIR = os.environ.get("NANOJEV_CKPT", "").strip() or os.path.join(REPO_DIR, "checkpoints", "NanoJev-unified")
if not os.path.isdir(CKPT_DIR) or not os.path.isfile(os.path.join(CKPT_DIR, "best.safetensors")):
    print("Скачиваем чекпойнт C-Tianyu/NanoJev (revision=unified-games-v1)...")
    snapshot_download(
        repo_id="C-Tianyu/NanoJev",
        revision="unified-games-v1",
        local_dir=CKPT_DIR,
        allow_patterns=["best.safetensors", "config.json", "tokenizer/*", "backbone_config/*"],
    )
print("Чекпойнт:", CKPT_DIR)
for f in ["config.json", "best.safetensors", "tokenizer", "backbone_config"]:
    print(" ", f, "OK" if os.path.exists(os.path.join(CKPT_DIR, f)) else "НЕТ")

In [ ]:
# 2.3 Загрузка DecisionPredictor (один раз на сессию).
#     Класс DecisionModel берётся из train_toy_decisions.py, веса — из best.safetensors.
from predict_toy_decisions import DecisionPredictor

if DEVICE != "cuda":
    print("Дальше нужна CUDA. В Colab: Runtime -> Change runtime type -> T4 GPU, затем «Выполнить всё».")
    NANOJEV_READY = False
else:
    try:
        nj = DecisionPredictor(CKPT_DIR, precision="bf16")     # стандартный путь (torch 2.x)
        _reason = "bf16"
    except Exception as e1:
        print("Прямая загрузка не удалась, пробуем disable_native_triton=True:")
        print("  ", type(e1).__name__, str(e1)[:200])
        nj = DecisionPredictor(CKPT_DIR, precision="bf16", disable_native_triton=True)
        _reason = "bf16 + disable_native_triton"
    NANOJEV_READY = True
    print("DecisionPredictor готов:", _reason)
    print("checkpoint.base_model:", nj.run_config.get("model"),
          "| max_length:", nj.limit, "| set_head:", nj.run_config["set_head"])

In [ ]:
# 2.4 Строим текстовый state и типизированный вопрос choice по фактическим классам.

# Характеристики классов (можно поправить под свой датасет) — семантика для модели.
CLASS_DESCRIPTIONS = {
    "IdleCharge":    "device is plugged in and idle or charging with low, stable power draw",
    "MixedBrowsing": "charging while doing light computer activity",
    "Browsing":      "active web browsing with moderate power draw",
    "VKAudio":       "audio or video streaming playback",
}
def class_desc(c: str) -> str:
    return CLASS_DESCRIPTIONS.get(c, f"home-appliance behaviour pattern {c}")

def chunk_stats(chunk: np.ndarray) -> dict:
    '''Те же 11 статистик, что уйдут в Random Forest (раздел 3) — единый вход.'''
    chunk = np.asarray(chunk, dtype=np.float64)
    return {
        "mean_mw": float(np.mean(chunk)),
        "std_mw": float(np.std(chunk)),
        "min_mw": float(np.min(chunk)),
        "max_mw": float(np.max(chunk)),
        "p25_mw": float(np.percentile(chunk, 25)),
        "p50_mw": float(np.percentile(chunk, 50)),
        "p75_mw": float(np.percentile(chunk, 75)),
        "range_mw": float(np.ptp(chunk)),
        "energy_g": float(np.sum(chunk ** 2) / 1e6),
        "mean_abs_diff": float(np.mean(np.abs(np.diff(chunk)))),
        "trend_mw_per_pt": float(np.polyfit(np.arange(len(chunk)), chunk, 1)[0]),
    }

def chunk_to_nanojev_state(chunk: np.ndarray, id_: int, n_points: int = 16) -> str:
    '''Числовой чанк -> текстовый state для NanoJev.'''
    s = chunk_stats(chunk)
    pts = np.round(chunk[::max(1, len(chunk) // n_points)][:n_points], 1)
    return (
        f"Power-consumption window #{id_} from a smart plug, P_RMS in mW, 90 samples.\n"
        f"Statistics: mean={s['mean_mw']:.1f}, std={s['std_mw']:.1f}, min={s['min_mw']:.1f}, "
        f"max={s['max_mw']:.1f}, p25={s['p25_mw']:.1f}, median={s['p50_mw']:.1f}, "
        f"p75={s['p75_mw']:.1f}, range={s['range_mw']:.1f}, energy_x1e-6={s['energy_g']:.2f}, "
        f"mean_abs_step={s['mean_abs_diff']:.2f}, trend_mw_per_pt={s['trend_mw_per_pt']:.3f}.\n"
        f"Sampled curve mW: " + ", ".join(str(float(v)) for v in pts) + "."
    )

def build_nanojev_questions(classes) -> dict:
    return {"pattern": {
        "type": "choice",
        "instructions": ("Which home-appliance behaviour pattern does this power-consumption "
                         "window belong to? Judge from the statistics and the sampled curve, "
                         "then choose exactly one of the listed patterns."),
        "criteria": {c: class_desc(c) for c in classes},
    }}

QUESTIONS_NJ = build_nanojev_questions(classes)
print("Кандидаты вопроса:", list(QUESTIONS_NJ["pattern"]["criteria"]))

def predict_one(text_state: str, qid="pattern") -> dict:
    '''Один state + один choice-вопрос -> словарь про ответ.'''
    payload = {"states": [{"id": "chunk", "state": text_state, "questions": QUESTIONS_NJ}]}
    res = nj.predict(payload)
    return res["states"][0]["answers"][qid]

# Пробный пример
demo = chunk_to_nanojev_state(X_test[0], 0)
print("=== state (первые 220 символов) ===")
print(demo[:220], "...")
print()
print("=== ответ ===")
ans = predict_one(demo)
print(json.dumps(ans, ensure_ascii=False, indent=2))
print("true:", test_labels[0])

In [ ]:
# 2.4b Пробный прогон на нескольких чанках из теста (визуальная проверка).
demo_idx = np.random.RandomState(1).choice(len(X_test), size=min(5, len(X_test)), replace=False)
for i in demo_idx:
    ans = predict_one(chunk_to_nanojev_state(X_test[i], int(i)))
    pred, conf = ans["choice"], float(ans["probabilities"][ans["choice"]])
    gt = test_labels[int(i)]
    ok = "OK" if pred == gt else "X"
    print(f"[{ok}] true={gt:<12s} pred={pred:<12s} conf={conf:.3f}")

In [ ]:
# 2.5 Массовая классификация тестовой выборки через NanoJev (пакетно).
MAX_N = 300          # ограничение размера для скорости; 0 = вся выборка
n = len(X_test) if MAX_N == 0 else min(MAX_N, len(X_test))
BATCH = 8            # сколько state-вопросов отправлять одним forward (память на T4)
print(f"Прогоняем {n} чанков батчами по {BATCH}...")

preds, confs = [], []
for start in tqdm(range(0, n, BATCH), desc="NanoJev predict"):
    idx = list(range(start, min(start + BATCH, n)))
    payload = {"states": [
        {"id": f"chunk_{int(i)}", "state": chunk_to_nanojev_state(X_test[i], int(i)),
         "questions": QUESTIONS_NJ} for i in idx]}
    res = nj.predict(payload)
    for i, st in zip(idx, res["states"]):
        ans = st["answers"]["pattern"]
        preds.append(ans["choice"])
        confs.append(float(ans["probabilities"][ans["choice"]]))

y_true_sub = le.inverse_transform(y_test[:n])
nj_acc = accuracy_score(y_true_sub, preds)
nj_f1 = f1_score(y_true_sub, preds, average="macro")
print("=== NanoJev zero-shot ===")
print(f"Accuracy : {nj_acc:.4f}")
print(f"Macro-F1 : {nj_f1:.4f}")
print(f"Средняя уверенность: {np.mean(confs):.4f}")

In [ ]:
# 2.6 Отчёт о качестве: классификация / матрица ошибок / точность по классам
print(classification_report(y_true_sub, preds))
cm = confusion_matrix(y_true_sub, preds)
plt.figure(figsize=(max(6, len(classes)), max(6, len(classes))))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
plt.yticks(range(len(classes)), classes)
plt.xlabel("Предсказано"); plt.ylabel("Истина")
for i in range(len(classes)):
    for j in range(len(classes)):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.title("NanoJev zero-shot: confusion matrix")
plt.show()

In [ ]:
# 2.7 Отбор по уверенности: accuracy по покрытию (проверка калиброванности zero-shot)
#     Отсекая низкие confidence — растёт ли точность? У «сырых» zero-shot моделей обычно нет.
if len(set(confs)) > 1:
    thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
    print("threshold | accuracy | coverage")
    for t in thresholds:
        mask = np.array(confs) >= t
        if mask.sum() == 0:
            continue
        acc_t = accuracy_score(np.array(y_true_sub)[mask], np.array(preds)[mask])
        print(f"{t:9.2f} | {acc_t:.4f} | {mask.mean():.3f}")

## 3. Сравнение с бейзлайном

In [ ]:
# 3.1 Бейзлайн: Random Forest на тех же статистических признаках (что и в state для NanoJev).
STAT_KEYS = ["mean_mw", "std_mw", "min_mw", "max_mw", "p25_mw", "p50_mw", "p75_mw",
             "range_mw", "energy_g", "mean_abs_diff", "trend_mw_per_pt"]

def stats_vector(chunk: np.ndarray) -> np.ndarray:
    s = chunk_stats(chunk)
    return np.array([s[k] for k in STAT_KEYS], dtype=np.float64)

X_feat = np.vstack([stats_vector(x) for x in tqdm(ndata, desc="Стат-признаки")])
print("Признаковое пространство:", X_feat.shape)

Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(
    X_feat, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
scaler = StandardScaler().fit(Xf_tr)
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(scaler.transform(Xf_tr), yf_tr)
rf_preds = rf.predict(scaler.transform(Xf_te))
rf_acc = accuracy_score(yf_te, rf_preds)
rf_f1 = f1_score(yf_te, rf_preds, average="macro")
print(f"RandomForest (stats): acc={rf_acc:.4f}  macro-F1={rf_f1:.4f}")

try:
    print(f"NanoJev zero-shot    : acc={nj_acc:.4f}  macro-F1={nj_f1:.4f}")
    delta = nj_acc - rf_acc
    print(f"Разница NanoJev - RF : {delta:+.4f}")
except NameError:
    print("Сначала выполните ячейку 2.5 (NanoJev predict). Обратите внимание: RF обучен на train, "
          "NanoJev zero-shot ничего не обучался — сравнение на одном тесте.")

## 4. Итоговая таблица: NanoJev zero-shot vs Random Forest

In [ ]:
# 4.1 Сводная таблица всех подходов (один и тот же сплит test из 1.4).
print("=" * 54)
print(f"{'Подход':<26}{'acc':>8}{'F1_macro':>10}")
print("-" * 54)
print(f"{'RandomForest stat (3.1)':<26}{rf_acc:>8.4f}{rf_f1:>10.4f}")
try:
    print(f"{'NanoJev zero-shot (2.5)':<26}{nj_acc:>8.4f}{nj_f1:>10.4f}")
    print("=" * 54)
    delta = nj_acc - rf_acc
    note = f"NanoJev хуже RF на {abs(delta):.3f}" if delta < 0 else f"NanoJev лучше RF на {delta:.3f}"
    print("Вывод:", note)
except NameError:
    print("(раздел 2 не выполнялся — нет CUDA)")
    print("=" * 54)

## 5. Выводы и ожидания

- **Все подходы оценены на одном тесте** — 30% стратифицированного сплита из 1.4.
- **Честное ожидание:** поскольку NanoJev обучался на игровых текстовых состояниях (Maze / Snake / ViZDoom), а не на временны́х рядах, zero-shot точность на `P_RMS`-окнах, скорее всего, **ниже** простого Random Forest на тех же признаках. Сами по себе признаки в тексте те же, что у RF — разница показывает **трансфер текстового decision-модели в числовую задачу**, а не качество признаков.
- **Когда сравнение интересно:** если NanoJev окажется недалеко от RF — это сильный сигнал о переносимости Jev-архитектуры; если провал — подтверждение, что модели этого класса нужно «видеть» предметную область (state в подходящем виде) либо дообучение/адаптация головы на наших данных (как делалось в `Jev_Laya_smartplug.ipynb`).
- **Калибровка:** см. 2.7 — если точность НЕ растёт с уверенностью, у zero-shot (как и у Laya из прошлого ноутбука) уверенность не является надёжной мерой качества; нужен RLCD-цикл.
- Полезные честные источники: [TypeSafe — Introducing System One Models](https://typesafe.ai/blog/introducing-system-one-models-and-jev), контракт ввода NanoJev — `docs/TYPESAFE_CONTRACT.md` в репозитории.